# Práctica 1: Estimación e Inferencia en Modelos de Regresión Lineales

## Modelo 1

Carguemos una base de datos de Woldridge "csal1"

In [ ]:
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pylab as plt
from wooldridge import *

dataWoo('ceosal1', description=True) #con description=True tenemos una descripción de las variables

Llamamos "datos" a los datos de esa base de datos, y dentro de tal, notamos por "y" al salario de los CEOS y a "X" la matriz de datos con columna de unos (constante) y "roe" (rendimiento medio del CEO):

In [ ]:
datos=dataWoo('ceosal1')

y=datos['salary']
X=sm.add_constant(datos['roe'])

## Estadísticos Descriptivos:

In [ ]:
media=np.mean(y)
Q1=np.quantile(y, 0.25)
Q3=np.quantile(y, 0.75)
Varianza=np.var(y)
DesviacionTipica=np.std(y)
Mediana=np.median(y)
histograma=plt.hist(y, bins='auto', rwidth=0.85)
plt.xlabel('y')
plt.ylabel('Frecuancia')
plt.title("Histrograma de y (salary)")
plt.show()
print(Q1, Mediana, Q3, DesviacionTipica, np.mean(y))

Ahora ajustamos el modelo $y = X\beta + u$ y extraemos un resumen del resultado:

In [ ]:
mco1 = sm.OLS(y, X).fit()
mco1.summary()

Existen otras formas de obtener las estimaciones del modelo:

In [ ]:
mco1=smf.ols('salary ~ roe', data=datos).fit()
mco1.summary()

Como el modelo solo tiene dos variables ("salary" y "roe") podemos dibujar los datos y la recta de regresión ajustada:

In [ ]:
beta=mco1.params
plt.plot(datos['roe'], y, 'o')
xmin=np.min(datos['roe'])
xmax=np.max(datos['roe'])
plt.plot([xmin,xmax], [beta[0]+beta[1]*xmin,beta[0]+beta[1]*xmax])
plt.show()

El modelo obtenido permite realizar algunas gráficas más que nos permitirán validar el modelo:

In [ ]:
sm.graphics.plot_regress_exog(mco1, 'roe')

Del modelo se pueden extraer otras medidas como:

* Valores predichos $\hat y$
* Residuos



In [ ]:
yhat=mco1.fittedvalues
e=mco1.resid
print(e)
np.mean(e)

* ANOVA

In [ ]:
table = sm.stats.anova_lm(mco1, typ=1)
print(table)

* Suma de Cuadrados Totales (SCT)

In [ ]:
mco1.centered_tss
sum((y-np.mean(y))**2)

* Suma de Cuadrados Explicada (SCE)

In [ ]:
mco1.ess
sum((mco1.fittedvalues-np.mean(y))**2)

* Suma de Cuadrados de los residuos (SCR)

In [ ]:
mco1.ssr
sum(e**2)

* $R^2$ y $R^2$-ajustado

In [ ]:
mco1.rsquared
mco1.rsquared_adj

* Valor $F_{exp}$ y de $F_{teo}$

In [ ]:
Fexp=mco1.fvalue
from scipy import stats
alpha=0.05
Fteo= stats.f.ppf(1-alpha, mco1.df_model, mco1.df_resid)
print(alpha, Fexp, Fteo)
alpha=0.10
Fteo= stats.f.ppf(1-alpha, mco1.df_model, mco1.df_resid)
print(alpha, Fexp, Fteo)

* Valores $t_{exp}$ y $t_{teo}$

In [ ]:
texp=mco1.tvalues
print("texp: ", texp)
alpha=0.05
tteo= stats.t.ppf(1-(alpha/2),mco1.df_resid)
print(alpha)
print(tteo)
alpha=0.10
tteo= stats.t.ppf(1-(alpha/2),mco1.df_resid)
print(alpha)
print(tteo)

* Intervalos de confianza de Estimadores

In [ ]:
mco1.conf_int()

* Estimación de la varianza de la pertrubación:

In [ ]:
beta=np.array(mco1.params)

sum(e**2)/(mco1.nobs-1)
sigmagorro=(np.dot(y.values, y.values)-np.dot(beta.T, np.dot(X.values.T,y.values)))/(mco1.nobs-1)

* Predicciones:

In [ ]:
from statsmodels.sandbox.regression.predstd import wls_prediction_std
epred, lb, ub = wls_prediction_std(mco1, weights=1)

In [ ]:
x0=5
beta[0]+beta[1]*x0
X0=sm.add_constant(x0)
mco1.predict(exog=dict(roe=x0))

## Cuestiones

1. Analizar el salario en función de los años de educación utilizando la base de datos **wage1**. Interpretar los coeficientes obtenidos. Dibuja ajustes y residuos.
3. Ajustar el modelo lineal del porcentaje de votos obtenido por el candidato A en base al porcentaje de gastos de campa\~na para tal candidato con la base de datos **vote1**. Comprobar que la media de los residuos es 0 y cómo se relaciona la media de $y$ con respecto a la media de $\hat{y}$.
4. Ajustar el modelo no lineal $\log(wage) = \beta _0 + \beta_1 {\rm educ} + u$ con la base de datos {\bf wage1} e interpretar los resultados obtenidos.
5. Usar la base de datos {\bf 401K} para estudiar la relación entre el porcentage de trabajadores activos que est\'an inscritos en el plan de pensiones (prate) y la tase de contribución al plan (mrate)- cantidad promedio con que la empresa contribuye al plan de cada trabajador  por cada dolar que aporta el trabajador. Seg\'un este modelo, ¿qué prate se predice para mrate=3.5? ? Cu\'anta variación de prate se explica por mrate?
6. Usando la base de datos **charity**:
 * ¿Cuál es el donativo (gift) promedio de esta muestra? ¿Qu\'e porcentage no dio donativo?
 * ¿Cuál es el promedio de envíos por año (mailsyear)?
 * Estimar ${\rm gift} = \beta_0 + \beta_1 {\rm mailsyear} + u$.
 * Si cada envío cuesta un florín, espera la beneficiencia obtener una ganancia neta por cada env\'io?
 * ¿Cu\'al es el menor donativo? Con el modelo de regresi'on, se puede predecir que gift=0?
7. Realiza el siguiente experimento:
 * Generar $500$ observaciones uniformes $[0,10]$. Calcular para esta muestra la media y la desviación t\'ipica: $x$
 * Generar $500$ errores seg\'un una normal $N(0,36)$: $e$. ? Es el promedio de la muestra 0? ¿Cu\'al es su desviación t\'ipica?
 * Ahora generar $y_i = 1 + 2x_i + e_i$ para $i=1, \ldots, 500$.
 * Estimar el modelo $y = \beta_0 + \beta_1 x + u$. Comparar el modelo real con el modelo ajustado.
 * Obtener $\hat{u}$ y probar si $\sum_{i=1}^{500} \hat{u}_i =0$ y que $\sum_{i=1}^{500} x_i \hat{u}_i =0$. ¿Ocurre lo mismo con $u$?
 * Generar de nuevo el modelo y comparar los resultados obtenidos. ¿Son iguales?